# RQ2_5 — Final Interpretation, Robustness & Validation

**Run after:** `RQ2_1_JIRA_Extraction.ipynb`, `RQ2_2_Reassignments.ipynb`, `RQ2_3_Analysis.ipynb`, and `RQ2_4_EDA.ipynb`.

## Purpose

This notebook is the robustness and interpretation layer for RQ2. It does **not replace RQ2_3**. The original OLS model remains the primary analysis, while this notebook tests whether the main conclusions are stable under alternative outcome and reassignment specifications.

### RQ2 focus

> **How do issue characteristics, including reassignment activity, relate to issue resolution time, and did these relationships differ between the pre-AI and AI-era periods?**

### Robustness strategy

1. Validate the analytical sample and project × era composition.
2. Re-estimate the original raw-resolution-time OLS model.
3. Add `log1p(resolution_time_days)` OLS because resolution time is highly right-skewed.
4. Add a Gamma GLM with log link as a distribution-sensitive robustness model.
5. Test reassignment count as continuous, binary, and categorical.
6. Test priority × era, comments × era, and reassignment × era interactions.
7. Compare effect sizes and model fit.
8. Produce a final evidence matrix and report-ready interpretation.

> **Causal caution:** AI era is a temporal proxy. Reassignment is an observed issue-history characteristic. Neither establishes that AI or reassignment caused a change in resolution time.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path

BASE = Path(".")

JIRA_FILE = "apache_jira_raw.csv"
REASSIGN_FILE = "num_reassignments_FULL.csv"  # UPDATED: full population (N=27,385), not the N=3,000 subsample
TARGET = "resolution_time_days"

print("RQ2_5 FINAL loaded.")
print("JIRA source:", JIRA_FILE)
print("Reassignment source:", REASSIGN_FILE)


RQ2_5 FINAL loaded.
JIRA source: apache_jira_raw.csv
Reassignment source: num_reassignments_FULL.csv


In [2]:
# IMPORTANT:
# num_reassignments_FULL.csv contains only:
#   issue_id, num_reassignments
# Resolution time, priority, comments, project and dates come from
# apache_jira_raw.csv, so the two real outputs must be merged exactly
# as in RQ2_3.

jira = pd.read_csv(BASE / JIRA_FILE)
reassign = pd.read_csv(BASE / REASSIGN_FILE)

merged = jira.merge(reassign, on="issue_id", how="inner")

merged["resolution_date_parsed"] = pd.to_datetime(
    merged["resolution_date"], errors="coerce", utc=True
)
merged["created_parsed"] = pd.to_datetime(
    merged["created"], errors="coerce", utc=True
)

merged["era"] = (
    merged["resolution_date_parsed"] >= pd.Timestamp("2023-01-01", tz="UTC")
).map({True: "ai_era", False: "pre_ai"})

merged["resolution_time_days"] = (
    merged["resolution_date_parsed"] - merged["created_parsed"]
).dt.total_seconds() / 86400

merged["era_binary"] = (merged["era"] == "ai_era").astype(int)

# Match RQ2_3 cleaning logic.
analysis = merged.dropna(
    subset=["resolution_time_days", "priority", "num_comments", "num_reassignments"]
).copy()

analysis = analysis[analysis["resolution_time_days"] >= 0].copy()

# Preserve the source terminology: num_comments, not num_comments.
analysis["num_comments"] = pd.to_numeric(
    analysis["num_comments"], errors="coerce"
)
analysis["num_reassignments"] = pd.to_numeric(
    analysis["num_reassignments"], errors="coerce"
)

analysis = analysis.dropna(
    subset=["resolution_time_days", "num_comments", "num_reassignments"]
).copy()

print(f"Merged sample: {len(merged)} issues")
print(f"Usable analytical sample: {len(analysis)}")
print("\nEra counts:")
display(analysis["era"].value_counts())

print("\nProject × era counts:")
if "project_name" in analysis.columns:
    display(
        analysis.groupby(["project_name", "era"])
        .size()
        .unstack(fill_value=0)
    )

print("\nColumns available for RQ2_5:")
print(analysis.columns.tolist())


Merged sample: 27385 issues
Usable analytical sample: 27385

Era counts:


era
pre_ai    22571
ai_era     4814
Name: count, dtype: int64


Project × era counts:


era,ai_era,pre_ai
project_name,,
CAMEL,4814,15361
HADOOP,0,7210



Columns available for RQ2_5:
['issue_id', 'project_name', 'created', 'resolution_date', 'priority', 'component', 'num_comments', 'num_reassignments', 'resolution_date_parsed', 'created_parsed', 'era', 'resolution_time_days', 'era_binary']


## 1. Sample and distribution validation

The original RQ2 analysis uses the reassignment-linked analytical dataset. This cell documents the final usable N and the extreme skew in resolution time.

A large difference between mean and median indicates that raw OLS should be complemented by a transformed/distribution-sensitive model.


In [3]:
dist = pd.DataFrame({
    "Statistic": ["N", "Mean", "Median", "Std Dev", "Minimum", "Maximum",
                  "90th percentile", "95th percentile", "99th percentile"],
    "Resolution time (days)": [
        len(analysis),
        analysis[TARGET].mean(),
        analysis[TARGET].median(),
        analysis[TARGET].std(),
        analysis[TARGET].min(),
        analysis[TARGET].max(),
        analysis[TARGET].quantile(.90),
        analysis[TARGET].quantile(.95),
        analysis[TARGET].quantile(.99)
    ]
})
display(dist.round(3))

print("\nReassignments: zero vs one or more")
analysis["reassignment_any"] = (analysis["num_reassignments"] >= 1).astype(int)
display(
    analysis["reassignment_any"]
    .value_counts()
    .rename(index={0:"0 reassignments",1:"≥1 reassignment"})
    .rename("N").to_frame()
)


,Statistic,Resolution time (days)
0,N,27385.000
1,Mean,62.280
2,Median,2.787
3,Std Dev,238.180
4,Minimum,0.000
5,Maximum,4634.097
6,90th percentile,109.963
7,95th percentile,283.095
8,99th percentile,1321.539



Reassignments: zero vs one or more


,N
reassignment_any,
≥1 reassignment,14403
0 reassignments,12982


## 2. Primary RQ2 model reproduction

This reproduces the conceptual structure of RQ2_3:

> resolution time ~ priority + comments + reassignments + AI era + priority×era + comments×era + reassignments×era

This raw-scale OLS model remains the **primary reference model**. The models below are robustness checks.


In [4]:
formula_primary = (
    f"{TARGET} ~ C(priority) + num_comments + num_reassignments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + num_reassignments:era_binary"
)

ols_raw = smf.ols(formula_primary, data=analysis).fit(cov_type="HC3")

print(ols_raw.summary())

raw_params = pd.DataFrame({
    "Coefficient": ols_raw.params,
    "p-value": ols_raw.pvalues,
    "CI low": ols_raw.conf_int()[0],
    "CI high": ols_raw.conf_int()[1]
})

display(raw_params.round(4))


                             OLS Regression Results                             
Dep. Variable:     resolution_time_days   R-squared:                       0.037
Model:                              OLS   Adj. R-squared:                  0.036
Method:                   Least Squares   F-statistic:                     41.51
Date:                  Fri, 21 Aug 2026   Prob (F-statistic):           4.68e-98
Time:                          06:48:28   Log-Likelihood:            -1.8823e+05
No. Observations:                 27385   AIC:                         3.765e+05
Df Residuals:                     27372   BIC:                         3.766e+05
Df Model:                            12                                         
Covariance Type:                    HC3                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 12
  warnings.warn('covariance of constraints does not have full '


,Coefficient,p-value,CI low,CI high
Intercept,-45.7378,0.0000,-55.7576,-35.7179
C(priority)[T.Critical],29.7164,0.0000,15.7828,43.6501
C(priority)[T.Major],68.5019,0.0000,60.0638,76.9400
C(priority)[T.Minor],66.6816,0.0000,57.0529,76.3102
C(priority)[T.Trivial],56.1540,0.0000,37.8714,74.4366
num_comments,2.7620,0.0000,2.3175,3.2064
num_reassignments,45.1555,0.0000,37.7602,52.5507
era_binary,-16.4863,0.0449,-32.5974,-0.3752
C(priority)[T.Critical]:era_binary,-18.6826,0.3072,-54.5414,17.1763
C(priority)[T.Major]:era_binary,2.3662,0.7081,-10.0207,14.7531


## 3. Log-transformed outcome robustness

Resolution time is strongly right-skewed. Therefore, the same predictor structure is estimated with:

`log1p(resolution_time_days)`

This does not replace the raw OLS result. It tests whether the direction and statistical evidence are stable when the long right tail is compressed.


In [5]:
analysis["log_resolution_time"] = np.log1p(analysis[TARGET].clip(lower=0))

formula_log = (
    "log_resolution_time ~ C(priority) + num_comments + num_reassignments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + num_reassignments:era_binary"
)

ols_log = smf.ols(formula_log, data=analysis).fit(cov_type="HC3")

log_params = pd.DataFrame({
    "Coefficient": ols_log.params,
    "p-value": ols_log.pvalues,
    "CI low": ols_log.conf_int()[0],
    "CI high": ols_log.conf_int()[1]
})

print("Log-outcome OLS:")
display(log_params.round(4))

print("Raw OLS R²:", round(ols_raw.rsquared, 4))
print("Log OLS R²:", round(ols_log.rsquared, 4))


Log-outcome OLS:


,Coefficient,p-value,CI low,CI high
Intercept,0.6708,0.0000,0.5655,0.7761
C(priority)[T.Critical],0.3186,0.0005,0.1382,0.4991
C(priority)[T.Major],0.4608,0.0000,0.3619,0.5596
C(priority)[T.Minor],0.4269,0.0000,0.3220,0.5317
C(priority)[T.Trivial],0.0647,0.4128,-0.0902,0.2197
num_comments,0.0622,0.0000,0.0579,0.0665
num_reassignments,0.7340,0.0000,0.6926,0.7755
era_binary,-0.1438,0.0587,-0.2928,0.0053
C(priority)[T.Critical]:era_binary,-0.1978,0.1776,-0.4853,0.0897
C(priority)[T.Major]:era_binary,0.1209,0.1045,-0.0251,0.2669


Raw OLS R²: 0.0366
Log OLS R²: 0.1938


## 4. Gamma GLM robustness

A Gamma generalized linear model with a log link provides an additional distribution-sensitive check for a positive, right-skewed duration outcome.

This model is included as a robustness analysis. It is not treated as evidence of causality.


In [6]:
# Gamma requires strictly positive outcomes.
gamma_df = analysis[analysis[TARGET] > 0].copy()

gamma_model = smf.glm(
    formula=formula_primary,
    data=gamma_df,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

gamma_params = pd.DataFrame({
    "Coefficient": gamma_model.params,
    "p-value": gamma_model.pvalues,
    "CI low": gamma_model.conf_int()[0],
    "CI high": gamma_model.conf_int()[1]
})

print(gamma_model.summary())
print("\nGamma GLM coefficients:")
display(gamma_params.round(4))


                  Generalized Linear Model Regression Results                   
Dep. Variable:     resolution_time_days   No. Observations:                27385
Model:                              GLM   Df Residuals:                    27372
Model Family:                     Gamma   Df Model:                           12
Link Function:                      Log   Scale:                          20.437
Method:                            IRLS   Log-Likelihood:            -1.1282e+05
Date:                  Fri, 21 Aug 2026   Deviance:                   1.7413e+05
Time:                          06:48:29   Pearson chi2:                 5.59e+05
No. Iterations:                      43   Pseudo R-squ. (CS):            0.01707
Covariance Type:              nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

,Coefficient,p-value,CI low,CI high
Intercept,2.2913,0.0000,1.9725,2.6100
C(priority)[T.Critical],0.6342,0.0245,0.0816,1.1869
C(priority)[T.Major],1.2821,0.0000,0.9675,1.5966
C(priority)[T.Minor],1.1776,0.0000,0.8507,1.5044
C(priority)[T.Trivial],1.1131,0.0000,0.6579,1.5683
num_comments,0.0351,0.0000,0.0289,0.0412
num_reassignments,0.4829,0.0000,0.3884,0.5773
era_binary,-1.0081,0.0342,-1.9410,-0.0752
C(priority)[T.Critical]:era_binary,-1.5844,0.3358,-4.8110,1.6421
C(priority)[T.Major]:era_binary,0.4622,0.3326,-0.4727,1.3972


## 5. Reassignment specification robustness

The original analysis treats `num_reassignments` as a continuous count. Because almost half of the analytical issues have zero reassignments and the maximum is six, we test three specifications:

1. **Continuous count**
2. **Binary:** 0 vs ≥1
3. **Categorical:** 0, 1, 2+

If the substantive conclusion is stable across these specifications, confidence in the reassignment finding increases.


In [7]:
analysis["reassignment_cat"] = pd.cut(
    analysis["num_reassignments"],
    bins=[-0.1, 0.5, 1.5, np.inf],
    labels=["0", "1", "2+"]
)

# Binary reassignment model
binary_formula = (
    f"{TARGET} ~ C(priority) + num_comments + reassignment_any + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + reassignment_any:era_binary"
)
ols_binary = smf.ols(binary_formula, data=analysis).fit(cov_type="HC3")

# Categorical reassignment model
cat_formula = (
    f"{TARGET} ~ C(priority) + num_comments + C(reassignment_cat) + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary + C(reassignment_cat):era_binary"
)
ols_cat = smf.ols(cat_formula, data=analysis).fit(cov_type="HC3")

reassignment_summary = pd.DataFrame({
    "Specification": [
        "Continuous count",
        "0 vs ≥1",
        "0 / 1 / 2+"
    ],
    "R²": [
        ols_raw.rsquared,
        ols_binary.rsquared,
        ols_cat.rsquared
    ],
    "Reassignment evidence": [
        f"coef={ols_raw.params.get('num_reassignments', np.nan):.4f}, p={ols_raw.pvalues.get('num_reassignments', np.nan):.4g}",
        f"coef={ols_binary.params.get('reassignment_any', np.nan):.4f}, p={ols_binary.pvalues.get('reassignment_any', np.nan):.4g}",
        "Joint/category comparison shown below"
    ]
})

display(reassignment_summary.round(4))
display(pd.DataFrame({
    "Coefficient": ols_cat.params,
    "p-value": ols_cat.pvalues,
    "CI low": ols_cat.conf_int()[0],
    "CI high": ols_cat.conf_int()[1]
}).loc[[x for x in ols_cat.params.index if "reassignment_cat" in x]].round(4))

# ---- ADDED: era-interaction terms (the piece missing from the original
# specification-robustness table -- main effects alone don't answer RQ2's
# actual era-moderation question) ----
print("\nEra-interaction terms (RQ2's actual moderation question):")
era_interaction_rows = pd.DataFrame({
    "Coefficient": [ols_binary.params.get("reassignment_any:era_binary", np.nan)] +
                   [ols_cat.params.get(t, np.nan) for t in ols_cat.params.index if "reassignment_cat" in t and "era_binary" in t],
    "p-value": [ols_binary.pvalues.get("reassignment_any:era_binary", np.nan)] +
               [ols_cat.pvalues.get(t, np.nan) for t in ols_cat.pvalues.index if "reassignment_cat" in t and "era_binary" in t],
}, index=["reassignment_any:era_binary"] + [t for t in ols_cat.params.index if "reassignment_cat" in t and "era_binary" in t])
display(era_interaction_rows.round(4))


,Specification,R²,Reassignment evidence
0,Continuous count,0.0366,"coef=45.1555, p=5.255e-33"
1,0 vs ≥1,0.0277,"coef=30.8914, p=1.342e-23"
2,0 / 1 / 2+,0.0456,Joint/category comparison shown below


,Coefficient,p-value,CI low,CI high
C(reassignment_cat)[T.1],20.0820,0.0000,14.2952,25.8687
C(reassignment_cat)[T.2+],169.3051,0.0000,139.9223,198.6879
C(reassignment_cat)[T.1]:era_binary,-29.9171,0.0000,-43.5202,-16.3140
C(reassignment_cat)[T.2+]:era_binary,132.8562,0.0647,-8.0984,273.8109



Era-interaction terms (RQ2's actual moderation question):


,Coefficient,p-value
reassignment_any:era_binary,-31.5718,0.0000
C(reassignment_cat)[T.1]:era_binary,-29.9171,0.0000
C(reassignment_cat)[T.2+]:era_binary,132.8562,0.0647


## 6. AI-era moderation

The key RQ2 moderation question is whether the relationship between reassignment activity and resolution time changed in the AI era.

The primary term is:

`num_reassignments × era_binary`

The same logic is applied to priority and comments.

A non-significant reassignment × era interaction means that the evidence does **not** support a statistically detectable change in the reassignment–resolution-time association across the two periods.


In [8]:
interaction_terms = [
    "C(priority):era_binary",
    "num_comments:era_binary",
    "num_reassignments:era_binary"
]

interaction_results = pd.DataFrame({
    "Interaction": interaction_terms,
    "Coefficient": [ols_raw.params.get(x, np.nan) for x in interaction_terms],
    "p-value": [ols_raw.pvalues.get(x, np.nan) for x in interaction_terms],
    "CI low": [ols_raw.conf_int().loc[x, 0] if x in ols_raw.params else np.nan for x in interaction_terms],
    "CI high": [ols_raw.conf_int().loc[x, 1] if x in ols_raw.params else np.nan for x in interaction_terms]
})

interaction_results["Significant at .05"] = interaction_results["p-value"] < 0.05
display(interaction_results.round(4))


,Interaction,Coefficient,p-value,CI low,CI high,Significant at .05
0,C(priority):era_binary,NaN,NaN,NaN,NaN,False
1,num_comments:era_binary,15.0745,0.0000,8.8194,21.3295,True
2,num_reassignments:era_binary,-16.7785,0.1198,-37.9199,4.3629,False


## 7. Effect-size and incremental explanatory value

The original RQ2 analysis reports the increase in R² after adding reassignment information. This section retains that logic and reports Cohen's f².

For the raw-scale OLS comparison:

- **Reduced model:** priority + comments + era + their era interactions
- **Extended model:** reduced model + reassignment + reassignment × era

Cohen's f² is:

`(R²_full - R²_reduced) / (1 - R²_full)`


In [9]:
reduced_formula = (
    f"{TARGET} ~ C(priority) + num_comments + era_binary "
    "+ C(priority):era_binary + num_comments:era_binary"
)

reduced_model = smf.ols(reduced_formula, data=analysis).fit(cov_type="HC3")

r2_reduced = reduced_model.rsquared
r2_full = ols_raw.rsquared
delta_r2 = r2_full - r2_reduced
f2 = delta_r2 / (1 - r2_full)

effect_summary = pd.DataFrame([{
    "Reduced R²": r2_reduced,
    "Full R²": r2_full,
    "Incremental R²": delta_r2,
    "Cohen f²": f2,
    "Reassignment coefficient": ols_raw.params.get("num_reassignments", np.nan),
    "Reassignment p-value": ols_raw.pvalues.get("num_reassignments", np.nan),
    "Reassignment × Era p-value": ols_raw.pvalues.get("num_reassignments:era_binary", np.nan)
}])

display(effect_summary.round(4))


,Reduced R²,Full R²,Incremental R²,Cohen f²,Reassignment coefficient,Reassignment p-value,Reassignment × Era p-value
0,0.0243,0.0366,0.0123,0.0128,45.1555,0.0,0.1198


## 8. Final evidence matrix

The evidence is intentionally separated into:

- primary raw-scale inference
- skew-robust outcome models
- reassignment specification robustness
- AI-era moderation
- incremental explanatory value

The final conclusion should distinguish the **main reassignment association** from the **AI-era moderation question**.


In [10]:
# ============================================================
# 8. FINAL EVIDENCE MATRIX — RQ2
# ============================================================

# Extract the key estimates and p-values safely
reassignment_coef = ols_raw.params.get(
    "num_reassignments", np.nan
)

reassignment_p = ols_raw.pvalues.get(
    "num_reassignments", np.nan
)

reassignment_era_coef = ols_raw.params.get(
    "num_reassignments:era_binary", np.nan
)

reassignment_era_p = ols_raw.pvalues.get(
    "num_reassignments:era_binary", np.nan
)

log_reassignment_coef = ols_log.params.get(
    "num_reassignments", np.nan
)

log_reassignment_p = ols_log.pvalues.get(
    "num_reassignments", np.nan
)

gamma_reassignment_coef = gamma_model.params.get(
    "num_reassignments", np.nan
)

gamma_reassignment_p = gamma_model.pvalues.get(
    "num_reassignments", np.nan
)

# ---- ADDED: era-interaction terms for log/Gamma/binary models. The
# original matrix only tested main-effect robustness for these three
# models, which does not answer RQ2's actual era-moderation question.
# The raw-OLS era-interaction (row 2 below) was the only era-interaction
# test in the original matrix. ----
log_reassignment_era_coef = ols_log.params.get(
    "num_reassignments:era_binary", np.nan
)
log_reassignment_era_p = ols_log.pvalues.get(
    "num_reassignments:era_binary", np.nan
)
gamma_reassignment_era_coef = gamma_model.params.get(
    "num_reassignments:era_binary", np.nan
)
gamma_reassignment_era_p = gamma_model.pvalues.get(
    "num_reassignments:era_binary", np.nan
)
binary_reassignment_era_coef = ols_binary.params.get(
    "reassignment_any:era_binary", np.nan
)
binary_reassignment_era_p = ols_binary.pvalues.get(
    "reassignment_any:era_binary", np.nan
)


# ------------------------------------------------------------
# Build the evidence matrix
# ------------------------------------------------------------

evidence = pd.DataFrame([

    # --------------------------------------------------------
    # 1. Primary reassignment main effect
    # --------------------------------------------------------
    {
        "Evidence": "HC3-robust OLS: reassignment main effect",

        "Estimate": reassignment_coef,

        "p-value": reassignment_p,

        "Supported?": reassignment_p < 0.05,

        "Interpretation": (
            "Reassignment is significantly associated with "
            "longer resolution time."
            if reassignment_p < 0.05
            else
            "No statistically significant association detected."
        )
    },


    # --------------------------------------------------------
    # 2. Reassignment × AI-era interaction
    # --------------------------------------------------------
    {
        "Evidence": "HC3-robust OLS: reassignment × AI era",

        "Estimate": reassignment_era_coef,

        "p-value": reassignment_era_p,

        "Supported?": reassignment_era_p < 0.05,

        "Interpretation": (
            "Statistically significant evidence that the "
            "reassignment–resolution-time relationship changed "
            "in the AI era."
            if reassignment_era_p < 0.05
            else
            "No statistically significant evidence that the "
            "reassignment–resolution-time relationship changed "
            "in the AI era."
        )
    },


    # --------------------------------------------------------
    # 3. Log-transformed outcome robustness
    # --------------------------------------------------------
    {
        "Evidence": "Log-outcome OLS: reassignment",

        "Estimate": log_reassignment_coef,

        "p-value": log_reassignment_p,

        "Supported?": log_reassignment_p < 0.05,

        "Interpretation": (
            "The reassignment association remains statistically "
            "significant after log-transforming resolution time."
            if log_reassignment_p < 0.05
            else
            "The reassignment association is not statistically "
            "significant after log transformation."
        )
    },


    # --------------------------------------------------------
    # 4. Gamma GLM robustness
    # --------------------------------------------------------
    {
        "Evidence": "Gamma GLM: reassignment",

        "Estimate": gamma_reassignment_coef,

        "p-value": gamma_reassignment_p,

        "Supported?": gamma_reassignment_p < 0.05,

        "Interpretation": (
            "The reassignment association remains statistically "
            "significant under a Gamma GLM."
            if gamma_reassignment_p < 0.05
            else
            "The reassignment association is not statistically "
            "significant under a Gamma GLM."
        )
    },


    # --------------------------------------------------------
    # 5. Log-outcome era-interaction (ADDED — tests era-moderation,
    #    not just the main-effect robustness of row 3)
    # --------------------------------------------------------
    {
        "Evidence": "Log-outcome OLS: reassignment × AI era",

        "Estimate": log_reassignment_era_coef,

        "p-value": log_reassignment_era_p,

        "Supported?": log_reassignment_era_p < 0.05,

        "Interpretation": (
            "Statistically significant evidence of era-moderation "
            "under the log-transformed outcome."
            if log_reassignment_era_p < 0.05
            else
            "No statistically significant evidence of era-moderation "
            "under the log-transformed outcome."
        )
    },

    # --------------------------------------------------------
    # 6. Gamma GLM era-interaction (ADDED)
    # --------------------------------------------------------
    {
        "Evidence": "Gamma GLM: reassignment × AI era",

        "Estimate": gamma_reassignment_era_coef,

        "p-value": gamma_reassignment_era_p,

        "Supported?": gamma_reassignment_era_p < 0.05,

        "Interpretation": (
            "Statistically significant evidence of era-moderation "
            "under the Gamma GLM."
            if gamma_reassignment_era_p < 0.05
            else
            "No statistically significant evidence of era-moderation "
            "under the Gamma GLM."
        )
    },

    # --------------------------------------------------------
    # 7. Binary-specification era-interaction (ADDED)
    # --------------------------------------------------------
    {
        "Evidence": "Binary spec (0 vs ≥1): reassignment × AI era",

        "Estimate": binary_reassignment_era_coef,

        "p-value": binary_reassignment_era_p,

        "Supported?": binary_reassignment_era_p < 0.05,

        "Interpretation": (
            "Statistically significant evidence of era-moderation "
            "under the binary reassignment specification."
            if binary_reassignment_era_p < 0.05
            else
            "No statistically significant evidence of era-moderation "
            "under the binary reassignment specification."
        )
    },

    # --------------------------------------------------------
    # 8. Incremental explanatory value
    # --------------------------------------------------------
    {
        "Evidence": "Incremental R² from reassignment",

        "Estimate": delta_r2,

        "p-value": np.nan,

        "Supported?": delta_r2 > 0,

        "Interpretation": (
            f"Adding reassignment increases explained variance "
            f"by {delta_r2:.4f}."
            if delta_r2 > 0
            else
            "Adding reassignment does not increase explained variance."
        )
    }

])


# ------------------------------------------------------------
# Display the final evidence matrix
# ------------------------------------------------------------

display(
    evidence.round(5)
)

,Evidence,Estimate,p-value,Supported?,Interpretation
0,HC3-robust OLS: reassignment main effect,45.15546,0.00000,True,Reassignment is significantly associated with ...
1,HC3-robust OLS: reassignment × AI era,-16.77848,0.11983,False,No statistically significant evidence that the...
2,Log-outcome OLS: reassignment,0.73403,0.00000,True,The reassignment association remains statistic...
3,Gamma GLM: reassignment,0.48288,0.00000,True,The reassignment association remains statistic...
4,Log-outcome OLS: reassignment × AI era,-0.36391,0.00000,True,Statistically significant evidence of era-mode...
5,Gamma GLM: reassignment × AI era,-0.25633,0.04600,True,Statistically significant evidence of era-mode...
6,Binary spec (0 vs ≥1): reassignment × AI era,-31.57177,0.00001,True,Statistically significant evidence of era-mode...
7,Incremental R² from reassignment,0.01230,NaN,True,Adding reassignment increases explained varian...


## 9. Report-ready interpretation (UPDATED — full population, N=27,385)

### What changed from the N=3,000 subsample

At full population scale, the picture is **more mixed than the original N=3,000
analysis**, not simply "more significant" or "less significant." Specifically:

- The **primary raw-scale model** (HC3-robust OLS on untransformed
  `resolution_time_days`) shows `num_reassignments x era_binary` as
  **not statistically significant** (p = 0.120), with a negative point estimate.
- However, **three of four robustness specifications** — the log-transformed
  outcome (p < 0.0001), the Gamma GLM (p = 0.046), and the binary
  reassignment specification (p < 0.0001) — all show a **statistically
  significant, consistently negative** reassignment x era interaction.
- The categorical specification is split: the "1 reassignment" tier is
  significant and negative (p < 0.0001), but the "2+" tier is only
  marginal (p = 0.065) and positive — likely reflecting the very small
  number of high-reassignment, AI-era issues driving that estimate.

### Why the raw-scale model likely understates this

Resolution time is extremely right-skewed (max = 4,634 days against a
median of 2.8 days; 99th percentile = 1,322 days). The raw-scale OLS model
is disproportionately influenced by this small number of extreme-duration
issues. The fact that the effect becomes significant and consistent as soon
as the outcome is transformed (log) or modeled with a distribution suited to
positive, skewed durations (Gamma GLM) — and stays consistent under an
outlier-insensitive binary specification — is a standard signature of a
raw-scale model being pulled around by its tail, not a null effect.

### Recommended wording

> **At full population scale (N=27,385), issue reassignment remains
> positively associated with longer resolution time (raw-scale coefficient
> = 45.2 days per reassignment, p < .001). Evidence on whether this
> relationship changed between the pre-AI and AI-era periods is mixed
> across specifications: the primary raw-scale model does not detect a
> statistically significant era interaction, but the interaction is
> significant and consistently negative under a log-transformed outcome,
> a Gamma GLM, and a binary reassignment specification. This pattern is
> consistent with the raw-scale model being influenced by extreme-duration
> outliers rather than with the absence of a real effect. On balance, the
> weight of the skew-robust evidence favors a real, moderate reduction in
* the reassignment-related resolution-time penalty during the AI era,
> though this should be reported as a qualified finding, not a clean
> significant/non-significant result.**

### Do not write

- "Reassignments cause delays."
- "AI caused faster/slower resolution."
- "AI reduced the impact of reassignment." (too strong — say the
  skew-robust evidence is *consistent with* a reduction, not that AI is
  established as the cause)
- "The model proves that AI changed issue resolution."
- "There is no evidence of an era effect" (this would only be accurate for
  the raw-scale model in isolation — it is not accurate once the robustness
  specifications are considered together)

### Preferred terminology

Use:

- "associated with"
- "estimated increase/decrease"
- "resolution time"
- "reassignment activity"
- "AI-era temporal proxy"
- "mixed evidence across specifications"
- "consistent with a raw-scale outlier effect"
- "does not establish causality"


## 10. Methodological decision rule (UPDATED — full population)

### RQ2 main-effect decision

The reassignment coefficient remains positive and statistically significant
(p < .001) across the raw OLS, log-outcome OLS, and Gamma GLM at full
population scale. The evidence for a robust association between
reassignment activity and longer resolution time is strengthened relative
to the N=3,000 subsample.

### RQ2 AI-era decision — requires qualified reporting, not a binary call

Unlike the original N=3,000 analysis (where `num_reassignments x era_binary`
was non-significant across the board), the full-population result is
**specification-dependent**:

- Not significant in the primary raw-scale model (p = 0.120)
- Significant and negative in the log-outcome model (p < 0.0001), the
  Gamma GLM (p = 0.046), and the binary specification (p < 0.0001)

Per this notebook's own hierarchy (primary model is the reference; robustness
checks assess stability), the correct interpretation is that the **primary
model's non-significance is not, by itself, sufficient grounds to report
"no evidence of an era effect."** When the majority of skew-robust
specifications agree with each other and disagree with the raw-scale model,
that disagreement should be reported explicitly, with the outlier-sensitivity
explanation, rather than defaulting to either the raw model's p-value alone
or the robustness models' p-values alone.

Recommended report language:

> **The raw-scale primary model does not detect a statistically significant
> reassignment x AI-era interaction, but this result is not stable: three of
> four robustness specifications (log-outcome, Gamma GLM, binary) show a
> significant, consistently negative interaction. This divergence is
> consistent with the raw-scale model being disproportionately influenced by
> a small number of extreme-duration issues. The full-population evidence is
> therefore reported as a qualified, specification-dependent finding rather
> than a clean significant/non-significant result.**

### Sampling terminology

The RQ2_2 notebook uses `DataFrame.sample()` without explicit strata
(confirmed in code) for its N=3,000 subsample. That subsample is now
**superseded** by the full-population extraction used throughout this
notebook (N=27,385, ~88% of the total population), so sampling-design
concerns (stratified vs. random) are no longer material to RQ2's primary
analysis.

### Causality rule

Neither reassignment nor AI era should be described as a causal mechanism
from these observational analyses alone.
